# backend

> One chat protocol over two very different engines.

Rishi runs a local litert model, synchronously, and speaks litert message dicts. fastllm
reaches every cloud provider, asynchronously, and speaks aidialog `Msg`/`Part`. They are
close cousins -- same lineage, same ordered-callback vocabulary, same
functions-with-docstrings tool contract -- which is why one `Backend` covers both without
a lowest common denominator worth apologising for.

Three things are genuinely different and are dealt with here rather than leaked upward:

**Async, from sync callers.** Both leela frontends already hand the assistant to a worker
thread, so the harness stays synchronous and the fastllm backend owns a private event loop
on a thread of its own. Not `asyncio.run` per turn, which would be simpler and wrong:
fastllm caches its httpx client across calls, and a client built on a loop that has since
closed fails on the next request. One long-lived loop also gives cancellation somewhere to
land -- `cancel()` cancels the task, which is the only way to stop a cloud turn at all,
since there is no local generation to interrupt.

**Approval.** Rishi gates tool calls in its own handler; fastllm has no gate until
`fastllm_hitl.apply()` puts one there. `set_approve` hides which of those is happening.

**Usage.** Rishi counts tokens by diffing the conversation's `token_count`; fastllm gets
real numbers, plus cost and cache hits, from the provider. `Usage` is the richer of the
two shapes, with the fields the local side cannot know left at zero -- because a harness
that carries two usage types ends up formatting them in two places.

Everything here degrades rather than raises. A missing engine is an ordinary state: the
model is a multi-gigabyte download on one side and an API key on the other, and an editor
must open without either.


In [ ]:
#| default_exp backend

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, threading
from dataclasses import dataclass
from ramabana.core import agent_err

In [ ]:
#| export
# How many tool-call rounds one turn may take. fastllm defaults to 2, which is a sensible
# number for a chat and far too few for a coding agent: reading a file, searching for a
# pattern and making an edit is already three. Rishi has no cap at all, which is the
# opposite mistake -- a small local model in a tool loop is exactly the thing that never
# stops -- so both are given the same explicit budget.
MAX_STEPS = 40

In [ ]:
#| export
@dataclass
class Usage:
    "Token and money accounting for a turn, in the richer of the two backends' shapes."
    model: str = ''
    input: int = 0
    output: int = 0
    total: int = 0
    cached: int = 0          # prompt tokens served from cache
    cache_write: int = 0     # prompt tokens written to cache
    reasoning: int = 0
    cost: float = 0.0
    turns: int = 0

    def __add__(self, o):
        if o is None: return self
        f = ('input', 'output', 'total', 'cached', 'cache_write', 'reasoning', 'cost', 'turns')
        return Usage(model=o.model or self.model, **{k: getattr(self, k) + getattr(o, k) for k in f})

    def __radd__(self, o): return self if o in (None, 0) else self.__add__(o)

    def __repr__(self):
        p = [f'{self.total:,} tok', f'in {self.input:,}', f'out {self.output:,}']
        if self.cached: p.append(f'cached {100*self.cached/max(self.input,1):.0f}%')
        if self.reasoning: p.append(f'thought {self.reasoning:,}')
        if self.cost: p.append(f'${self.cost:.4f}')
        if self.model: p.append(self.model.split('/')[-1])
        return ' · '.join(p)

    def dict(self): return dict(self.__dict__)

In [ ]:
#| export
# ---------------------------------------------------------------------------
# driving async code from a synchronous harness
# ---------------------------------------------------------------------------
class _Loop:
    """A private event loop on a daemon thread, started once and kept.

    The alternative -- `asyncio.run` per call -- reads better and breaks fastllm, whose
    `mk_client` is `flexicache`d: the second turn would reuse an httpx client whose loop
    had closed under it. Keeping one loop also means an in-flight turn is a task that can
    be cancelled from another thread, which is what `Backend.cancel` needs.
    """

    def __init__(self):
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self._run, daemon=True, name='leela-agent-loop')
        self.thread.start()

    def _run(self):
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()

    def submit(self, coro):
        "Run `coro` to completion, blocking this thread. Returns a `(future, result_fn)` pair."
        return asyncio.run_coroutine_threadsafe(coro, self.loop)

    def close(self):
        if self.loop.is_closed(): return
        self.loop.call_soon_threadsafe(self.loop.stop)
        self.thread.join(timeout=5)
        try: self.loop.close()
        except Exception: pass

In [ ]:
#| export
_loop = None

In [ ]:
#| export
_loop_lock = threading.Lock()

In [ ]:
#| export
def shared_loop():
    "The one loop every cloud backend shares. Built on first use; there is no reason for two."
    global _loop
    with _loop_lock:
        if _loop is None or _loop.loop.is_closed(): _loop = _Loop()
        return _loop

In [ ]:
#| export
# ---------------------------------------------------------------------------
# the protocol
# ---------------------------------------------------------------------------
class Backend:
    """One conversation with one model.

    Subclasses implement `_start`, `_send`, `_stream`, `_oneshot` and `_usage`; everything
    public here is shared, including the rule that a backend which failed to start is not
    an error but a state (`ready` is False and `note` says why).
    """

    kind = '?'

    def __init__(self, spec, sp='', tools=(), approve=None, tool_max_len=None, shared=False, **kw):
        self.spec, self.sp, self.tools, self.approve = spec, sp, list(tools), approve
        self.tool_max_len, self.kw = tool_max_len, kw
        # `shared` marks a backend that borrowed someone else's engine, so closing it
        # releases its conversation and leaves the engine for whoever actually owns it.
        self.shared = shared
        self.chat = None
        self.use = Usage(model=spec.model_id)
        self.note = 'not started'
        # Everything that went wrong and was not raised to a caller who could show it: a
        # cheap job that failed, an engine that complained on stderr, a turn that came back
        # empty. `note` is one line and gets overwritten by the next thing; this keeps.
        self.problems = []
        # What the engine last wrote to stdout/stderr that looked like a complaint. Only a
        # native backend fills this in; see `leela.agent.native`.
        self.last_native = ''
        self._tried = False
        self.lock = threading.Lock()

    def problem(self, text):
        """Record something the user should know, and return it.

        Silence is the failure mode this exists for. A completion, a classification or a
        compaction that fails returns `''` to its caller by design -- none of them can
        usefully raise -- and until this list existed, `''` was the entire report.
        """
        text = (text or '').strip()
        if not text: return ''
        if not self.problems or self.problems[-1] != text: self.problems.append(text)
        del self.problems[:-20]
        return text

    def _failed(self, what, e):
        "One failure, with whatever the native engine said on its way out, recorded and returned."
        native = getattr(e, 'native_output', '') or ''
        self.note = f'{self.spec.name} {what} ({agent_err(e)})' + (f' — {native}' if native else '')
        return self.problem(self.note)

    # -- lifecycle -----------------------------------------------------------
    @property
    def ready(self): return self.chat is not None

    @property
    def busy(self): return self.lock.locked()

    def start(self):
        "Build the chat, once. Returns it, or None with `note` explaining why not."
        if self._tried: return self.chat
        self._tried = True
        try:
            self.chat = self._start()
            self.note = f'{len(self.tools)} tools'
        except Exception as e:
            self.chat = None
            self._failed('unavailable', e)
        return self.chat

    def retry(self):
        "Forget a previous failure, so a model that has since downloaded or been keyed is picked up."
        self._tried, self.chat = False, None
        return self.start()

    def set_approve(self, approve):
        "Change the approval policy, including mid-session. Applied to a live chat when there is one."
        self.approve = approve
        if self.chat is not None: self._set_approve(approve)
        return self

    def close(self):
        if self.chat is not None:
            try: self._close()
            except Exception: pass
            self.chat = None

    def cancel(self):
        "Stop the turn in flight. Returns whether there was anything to stop."
        if self.chat is None: return False
        try: return bool(self._cancel())
        except Exception: return False

    # -- one turn ------------------------------------------------------------
    def send(self, msg, **kw):
        "One turn. Returns the reply text, or the reason there isn't any."
        if self.start() is None: return self.note
        with self.lock:
            try:
                out = self._send(msg, **kw)
                self.use = self._usage()
                return out or self._empty()
            except Exception as e:
                return self._failed('failed', e)

    def stream(self, msg, **kw):
        "One turn as an iterator of markdown chunks. Falls back to a single chunk on failure."
        if self.start() is None:
            yield self.note
            return
        with self.lock:
            n = 0
            try:
                for c in self._stream(msg, **kw):
                    n += len(c or '')
                    yield c
                self.use = self._usage()
                # A turn that produced nothing is a failure that did not raise -- the usual
                # cause is the engine refusing the request in native code, where no
                # exception is available to catch. Whatever it said is worth more than the
                # blank pane that used to be the whole report.
                if not n and (why := self._empty(strict=True)): yield why
            except Exception as e:
                yield f'\n\n{self._failed("failed", e)}'

    def _empty(self, strict=False):
        """What to say about a turn that came back with nothing.

        `last_native` is the point: on a local engine an empty reply usually means the
        native layer refused the request and said so on a file descriptor, so the answer to
        "why is the pane blank" is sitting right there. Without it this is still better
        than nothing -- it at least says the model returned nothing, rather than showing an
        empty box and a status line that says everything is fine.
        """
        why = (f'{self.spec.name} returned nothing'
               + (f' — {self.last_native}' if self.last_native else ''))
        return self.problem(why) if strict else ((f'({why})') if self.last_native else '(no reply)')

    def oneshot(self, prompt, sp='', max_tokens=None):
        """A question in a throwaway conversation on this engine, with no tools.

        The expensive part of a backend is the engine, not the conversation, so a
        completion, a classification or a summary borrows the loaded one and opens a
        conversation it then discards. That is what keeps them stateless -- a suggestion
        about the code on screen is a question, not a turn -- and keeps them out of the
        conversation the user is actually having.
        """
        if self.start() is None: return ''
        if not self.lock.acquire(blocking=False): return ''
        try: return self._oneshot(prompt, sp, max_tokens) or ''
        except Exception as e:
            # Returning '' is right -- none of the callers can do anything with an
            # exception -- but returning '' *and saying nothing* is how a compaction that
            # could not run, or a completion the engine refused, became invisible.
            self._failed('one-shot failed', e)
            return ''
        finally: self.lock.release()

    # -- context -------------------------------------------------------------
    @property
    def hist(self):
        "The conversation so far, in whatever shape the engine keeps it."
        return getattr(self.chat, 'hist', []) if self.chat is not None else []

    def replace_hist(self, summary, keep=()):
        """Throw away the conversation and start again from `summary` plus `keep`.

        What compaction actually is, once the summarizing is done. Kept behind a method
        because the two engines mean very different things by "history": fastllm's is a
        plain list that can be reassigned, while rishi's lives inside a litert conversation
        that owns a KV cache, so the only honest way to shorten it is to build a new one.
        """
        if self.chat is None: raise RuntimeError('nothing to compact: the model is not running')
        self._replace_hist(summary, list(keep))
        return self

    def count_tokens(self, text):
        "Tokens in `text`. Exact where the engine has a tokenizer, estimated where it does not."
        return max(1, (len(text or '') + 3) // 4)

    @property
    def used_tokens(self):
        "Roughly how full the context is right now, in tokens."
        return self.use.total

    @property
    def pct_full(self): return self.used_tokens / max(self.spec.ctx, 1)

    # -- subclass hooks ------------------------------------------------------
    def _start(self): raise NotImplementedError
    def _send(self, msg, **kw): raise NotImplementedError
    def _stream(self, msg, **kw): raise NotImplementedError
    def spawn(self, sp='', tools=(), **kw):
        """A second, independent conversation on the *same* engine.

        What a sub-agent runs in. The engine is the expensive thing -- gigabytes locally, a
        client and its connection pool remotely -- and the conversation is nearly free, so
        delegation should never mean loading a second model. The child gets its own
        history, its own tool set and no approval policy inherited from the parent.
        """
        raise NotImplementedError

    def _oneshot(self, prompt, sp, max_tokens): raise NotImplementedError
    def _replace_hist(self, summary, keep): raise NotImplementedError
    def _usage(self): return self.use
    def _set_approve(self, approve): self.chat.approve = approve
    def _cancel(self): return False
    def _close(self): self.chat.close()

In [ ]:
#| export
# ---------------------------------------------------------------------------
# local: rishi over litert
# ---------------------------------------------------------------------------
class RishiBackend(Backend):
    """A local litert model through rishi. Synchronous all the way down, so this is mostly pass-through.

    Except for one thing that is not pass-through at all: litert is C++, and when it
    refuses a request it says so on a file descriptor and returns, without raising. Every
    call into the engine therefore runs inside `native.captured`, which tees that output
    and keeps the tail -- so "input token IDs exceed the maximum number of tokens 4096, got
    5092" ends up on the status bar instead of scrolling past in the terminal that happened
    to launch the server.
    """

    kind = 'rishi'

    def _native(self, fn, *a, **kw):
        "Run one engine call, keeping whatever it printed. Exceptions carry it out on `native_output`."
        from ramabana.native import capture
        try:
            out, said = capture(fn, *a, **kw)
        except Exception:
            self.last_native = ''
            raise
        self.last_native = said
        if said: self.problem(f'{self.spec.name}: {said}')
        return out

    def _start(self):
        from rishi.core import Chat
        return self._native(Chat, sp=self.sp, tools=self.tools, model_id=self.spec.model_id,
                            approve=self.approve, tool_max_len=self.tool_max_len,
                            ctx_limit=self.spec.ctx, **self.kw)

    def spawn(self, sp='', tools=(), **kw):
        "A second conversation on this engine. `engine=` is rishi's own way of saying 'do not load another'."
        if self.start() is None: raise RuntimeError(self.note)
        return RishiBackend(self.spec, sp=sp, tools=tools, tool_max_len=self.tool_max_len,
                            shared=True, engine=self.chat.engine, **kw)

    def _close(self):
        # Rishi's exit stack owns both the conversation and the engine, and closing a
        # borrowed engine would take the parent's model down with the child's question.
        if self.shared:
            try: self.chat.conv.close()
            except Exception: pass
        else: self.chat.close()

    def _send(self, msg, **kw):
        from rishi.core import resp_text
        return resp_text(self._native(self.chat, msg, **kw))

    def _stream(self, msg, **kw):
        # rishi's `_stream` already yields formatted markdown, so there is nothing to add
        # but the capture, which has to span the whole iteration: the engine does its
        # complaining on the first send, long before the first chunk that never comes.
        from ramabana.native import captured
        with captured() as cap:
            yield from self.chat(msg, stream=True, **kw)
        self.last_native = cap.problems
        if cap.problems: self.problem(f'{self.spec.name}: {cap.problems}')

    def _oneshot(self, prompt, sp, max_tokens):
        from rishi.core import mk_msg, resp_text
        pre = [{'role': 'system', 'content': sp}] if sp else None

        def run():
            with self.chat.engine.create_conversation(messages=pre) as conv:
                return resp_text(conv.send_message(mk_msg(prompt), max_output_tokens=max_tokens))
        return self._native(run)

    def _usage(self):
        u = self.chat.use
        return Usage(model=self.spec.model_id, input=u.prompt_tokens, output=u.completion_tokens,
                     total=u.total_tokens, turns=u.n)

    def count_tokens(self, text):
        if self.chat is None: return super().count_tokens(text)
        try: return self.chat.count_tokens(text or '')
        except Exception: return super().count_tokens(text)

    @property
    def used_tokens(self):
        try: return self.chat.token_count if self.chat is not None else 0
        except Exception: return self.use.total

    def _replace_hist(self, summary, keep):
        """Rebuild the litert conversation around a shorter history.

        There is no way to shorten a litert conversation in place -- it owns a KV cache
        keyed to the tokens already in it -- so the conversation is rebuilt on the *same*
        engine, which is where all the cost is. The old one is released explicitly rather
        than left to `Chat`'s exit stack, which would otherwise hold the whole compacted
        context alive for the rest of the session.
        """
        from rishi.core import ChatToolHandler, mk_msgs
        c, old = self.chat, self.chat.conv
        hist = mk_msgs([summary] + list(keep))
        pre = ([{'role': 'system', 'content': c.sp}] if c.sp else []) + hist
        c.conv = c.engine.create_conversation(messages=pre or None, tools=list(c.tools) or None,
                                              tool_event_handler=ChatToolHandler(c))
        c.hist = hist
        c._tc0 = 0
        for m in ('close', '__exit__'):
            try:
                getattr(old, m)(None, None, None) if m == '__exit__' else getattr(old, m)()
                break
            except Exception: continue

    def _cancel(self):
        self.chat.cancel()
        return True

In [ ]:
#| export
# ---------------------------------------------------------------------------
# cloud: fastllm over every provider
# ---------------------------------------------------------------------------
class FastllmBackend(Backend):
    """A cloud model through fastllm, driven from synchronous code.

    `cache` is on by default and is close to free money for this workload: the system
    prompt is large and about to grow a skill index, and the notebook context above a
    prompt cell is stable across a run of questions. fastllm only applies it to models
    that support it, so leaving it on costs nothing where it does not apply.
    """

    kind = 'fastllm'

    def __init__(self, *args, cache=True, max_steps=MAX_STEPS, think=None, **kw):
        self.cache, self.max_steps, self.think = cache, max_steps, think
        super().__init__(*args, **kw)
        self._task = None
        self._patched = False

    def _start(self):
        from ramabana.fastllm_hitl import apply, note
        from fastllm.chat import AsyncChat
        self._patched = apply()
        c = AsyncChat(self.spec.model_id, sp=self.sp, tools=self.tools,
                      cache=self.cache, cache_idxs=[0, -1], **self.kw)
        c.approve = self.approve
        if self.approve is not None and not self._patched and not self.shared:
            # Refusing to run is the only honest option: a gated backend that silently
            # stops gating is worse than one that will not start, because the failure is
            # invisible exactly when it matters.
            raise RuntimeError(f'tool approval was requested but could not be installed: {note()}')
        return c

    def spawn(self, sp='', tools=(), **kw):
        "Trivially cheap here: an `AsyncChat` is a message list and a schema list, and the client is cached."
        return FastllmBackend(self.spec, sp=sp, tools=tools, cache=self.cache,
                              max_steps=self.max_steps, shared=True, **kw)

    def _run(self, coro):
        "Run a coroutine on the shared loop, keeping the task so `cancel` has something to hit."
        self._task = shared_loop().submit(coro)
        try: return self._task.result()
        finally: self._task = None

    def _kw(self, kw):
        d = dict(max_steps=self.max_steps, **kw)
        if self.think and 'think' not in d: d['think'] = self.think
        return d

    def _send(self, msg, **kw):
        from fastllm.chat import contents
        from aidialog.msg_parts import PartType
        res = self._run(self.chat(_fl_msg(msg), **self._kw(kw)))
        m = contents(res)
        if not m: return ''
        return ''.join(p.text or '' for p in m.content if p.type == PartType.text)

    def _stream(self, msg, **kw):
        """Pump fastllm's async chunk stream out to a synchronous caller.

        A queue and a thread rather than anything cleverer, because the consumer is a
        Textual worker or a FastHTML handler that wants a plain iterator, and the producer
        is an async generator on another loop. `_task` is set so a stream can be cancelled
        like a turn.
        """
        import queue
        from fastllm.chat import StreamFormatter
        q, done, fmt = queue.Queue(), object(), StreamFormatter()

        async def pump():
            try:
                async for o in await self.chat(_fl_msg(msg), stream=True, **self._kw(kw)):
                    q.put(('o', o))
            except Exception as e: q.put(('e', e))
            finally: q.put((done, None))

        self._task = shared_loop().submit(pump())
        try:
            while True:
                k, v = q.get()
                if k is done: break
                if k == 'e': raise v
                if (s := fmt.format_item(v)): yield s
        finally:
            self._task = None

    def _oneshot(self, prompt, sp, max_tokens):
        from fastllm.acomplete import acomplete
        from fastllm.chat import mk_msgs, contents
        from aidialog.msg_parts import PartType
        res = self._run(acomplete(mk_msgs([prompt]), self.spec.model_id, system=sp,
                                  max_tokens=max_tokens or 1024))
        m = contents(res)
        return '' if not m else ''.join(p.text or '' for p in m.content if p.type == PartType.text)

    def _usage(self):
        u = self.chat.use
        return Usage(model=u.model or self.spec.model_id, input=u.prompt_tokens, output=u.completion_tokens,
                     total=u.total_tokens, cached=u.cached_tokens, cache_write=u.cache_creation_tokens,
                     reasoning=u.reasoning_tokens, cost=u.cost, turns=1)

    def _replace_hist(self, summary, keep):
        """Reassign the list. fastllm keeps no cache keyed to it, so this really is that easy.

        The one care needed is the join: if the tail already starts at a user turn --
        which `Compactor` guarantees -- the summary is folded *into* that message rather
        than sent as a second consecutive user message, which providers variously merge,
        reject, or quietly treat as two turns.
        """
        from fastllm.chat import mk_msg, mk_msgs
        from aidialog.msg_parts import Part, PartType
        keep = list(keep)
        if keep and getattr(keep[0], 'role', None) == 'user':
            keep[0].content = [Part(PartType.text, summary)] + list(keep[0].content)
            self.chat.hist = mk_msgs(keep)
        else:
            self.chat.hist = mk_msgs([mk_msg(summary, role='user')] + keep)

    @property
    def used_tokens(self):
        "The last request's input plus its output: what the next request will be built on."
        u = getattr(self.chat, 'last_req_use', None) if self.chat is not None else None
        return (u.prompt_tokens + u.completion_tokens) if u else self.use.total

    def _cancel(self):
        t = self._task
        return bool(t is not None and t.cancel())

    def _close(self):
        pass   # the shared loop outlives any one chat, and fastllm's client cache is global

In [ ]:
#| export
def _fl_msg(msg):
    "leela hands multimodal turns over as `[image_bytes, text]`; fastllm takes that shape directly."
    return msg

In [ ]:
#| export
# ---------------------------------------------------------------------------
def make_backend(spec, **kw):
    "The backend for a resolved `ModelSpec`."
    cls = {'rishi': RishiBackend, 'fastllm': FastllmBackend}.get(spec.backend)
    if cls is None: raise KeyError(f'no backend for {spec.backend!r}')
    return cls(spec, **kw)

## Tests


In [ ]:
# `Usage` is the richer of the two engines' shapes, so a harness that mixes a local turn
# and a cloud one carries one type rather than two. rishi's `UsageStats` was given `cost`
# and `model` for exactly this reason.
a = Usage(model='m', input=100, output=10, total=110, turns=1)
b = Usage(model='m', input=50, output=5, total=55, cost=0.002, turns=1)
print(a + b)
assert (a + b).total == 165 and (a + b).turns == 2
assert sum([a, b], None).input == 150

In [ ]:
# A turn can be driven end to end with no model at all, which is what makes every test
# above this line cheap. `ramabana.testing.FakeBackend` is the same double leela's suite uses.
from ramabana.testing import FakeBackend, SPEC
be = FakeBackend(SPEC, replies=['first answer', 'second answer'])
be.start()
print(be.send('hello'))
print(be.send('again'))
print('history:', [m['role'] for m in be.hist])
assert be.send('third') == '(done)'   # the script runs out, it does not raise